# AI Modified Workflow

For this workflow we use the following prompt to modify the `scale_model_size.py` notebook:

Fit a simple linear model to the data, report the slope and R-squared, annotate the plot with the fit results.

---

Copilot succesfully added the linear fit line and reported on the slope and R-squared. It attempted to self-validate but failed to due so (because I hadn't actually gathered any data for it yet).

---
---
---

# Configuration

## Import workflows module

In [ ]:
from utils.workflows import *

## Global params

Users can modify these top-level parameters to alter the behavior of this workflow.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------------
# !!! DO NOT MODIFY THE CODE BELOW   !!!
# !!!  (Modify in the next section)  !!!
# ---------------------------------------------------------------------------------------------------------------------

# So that can you can maintain the defaults, we suggest you don't directly edit
# the parameters inline here but rather overwite values at the bottom of this
# cell.

# We'll store our containers and benchmark results under the specified directory
# (it will be created if it doesn't already exist).
import os
if 'user_customExperimentsDir' in globals():
    baseDir = f'{user_customExperimentsDir}/scale_model_size'
else:
    baseDir=f'{os.getenv("HOME")}/workflows/scale_model_size'

# Run using an SST in the specified container. To find containers to use see the
# container factory at https://github.com/hpc-ai-adv-dev/sst-container-factory
# Prebuild containers are available at https://github.com/orgs/hpc-ai-adv-dev/packages
container_url  = 'ghcr.io/hpc-ai-adv-dev/sst-core:master-latest'
container_name = None # DO NOT MODIFY THIS LINE: Variable will be assigned after we download the container
                      # We include it here to document what global variables are available throughout the
                      # notebook.

# The benchmark will be cloned from the specified repository. We assume the
# benchmark itself is in the 'benchmarkPath' directory within the repos.  We
# assume building the benchmark is a matter of running 'make' in that directoy.
benchmarkRepos='https://github.com/hpc-ai-adv-dev/sst-benchmarks.git'
benchmarkPath='phold'

# Run the benchmark on a single node, increasing the numbers of components with each trial
num_comps_per_trial  = [1_000_000, 2_000_000, 3_000_000, 4_000_000, 5_000_000]

# This command will be run prior to launching a job. The command will be run
# from within the benchmark directory and execution occurs within the worklaunch
# loop so it may be parameterized by the trial parameters if needed.
prestart_cmd_template = ''

# Indicates what arguments should be passed to sst and the benchmark each run 
# Note: {width} and {height} will be replaced with the appropriate values for
# each run, based on the number of nodes and components per node
sst_args_template   = '--print-timing-info=3 --parallel-load=SINGLE ./phold_dist.py'
bmark_args_template = '--width {width} --height {height}'

# Additional arguments to pass when launching jobs with srun. For example the
# partition name or --qos=high for higher priority in the queue.
additional_srun_args = ''

# Several of the setup steps will avoid rerunning if they have previously been run. Append to this
# list to indicate when you want to force a step to be reproduced.
#
# VALID VALUES ARE:
#   'ALL'     
#   'DOWNLOAD_CONTAINERS' 
#   'DOWNLOAD_BENCHMARKS' 
#   'BUILD_BENCHMARKS'      Note: we always rerun make, if this is set we will also run 'make clean' before rebuilding
force = []

# ---------------------------------------------------------------------------------------------------------------------
# Overwite parameters below this line to customize the workflow: 
# ---------------------------------------------------------------------------------------------------------------------

num_comps_per_trial  = [1_000, 2_000, 3_000, 4_000, 5_000]
print(baseDir)

## Environment

In [ ]:
set_workflow_log(f'{baseDir}/workflow.log')
run_cmd(f'e4s-cl profile edit --add-files {baseDir}')

## Download containers

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_CONTAINERS' in force

container_name = download_custom_container(container_url, force=_force)

## Download benchmarks

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_BENCHMARKS' in force

if not os.path.exists(f'benchmarks') or _force:
    run_cmd(f"git clone {benchmarkRepos} benchmarks")
    run_cmd(f"e4s-cl profile edit --add-files {baseDir}/benchmarks/{benchmarkPath}")
else:
    print(f"Benchmarks from {benchmarkRepos} have already been downloaded, skipping download.")

## Build benchmarks 

In [ ]:
_force = 'ALL' in force or 'BUILD_BENCHMARKS' in force

cd(f"{baseDir}/benchmarks/{benchmarkPath}")
run_cmd('touch sstsimulator.conf')
_cmd = 'make' if not _force else 'make clean; make'
run_in_container(_cmd,
    f'{baseDir}/{container_name}',
    additional_apptainer_args=f'--bind sstsimulator.conf:{os.getenv("HOME")}/.sst/sstsimulator.conf')
cd(baseDir)

# Run

## Start jobs

In [13]:
import math, shutil, os

runDisplay = SafeDisplay(display_handle = display('', display_id="run_disp"))

# Setup directory to store results in
run_dir = f'{baseDir}/runs/'
if os.path.exists(run_dir):
    shutil.rmtree(run_dir)
os.makedirs(run_dir, exist_ok=True)

cd(f"{baseDir}/benchmarks/{benchmarkPath}")

# Deploy jobs
for approx_size in num_comps_per_trial:
    width  = int(math.sqrt(approx_size))
    height = width
    size = width*height

    if prestart_cmd_template is not None and prestart_cmd_template != '':
        run_cmd(prestart_cmd_template.format(width=width, height=height, size=size))

    full_sst_args_template = f'{sst_args_template} -- {bmark_args_template}'
    sst_args = full_sst_args_template.format(width=width, height=height, size=size)

    launch_and_log_sst(
        image        = f'{baseDir}/{container_name}',
        srun_args    = f'-N 1 --job-name={benchmarkPath.lower()}_{size} {additional_srun_args}',
        sst_args     = sst_args,
        log_file     = f'{run_dir}/size_{size}',
        config_path  = f'{baseDir}/benchmarks/{benchmarkPath}/sstsimulator.conf',
        safe_display = runDisplay)

cd(f"{baseDir}")

[+] Using selected profile default
initializing simulation on rank 0 of 1 with 1 threads
[+] Using selected profile default
initializing simulation on rank 0 of 1 with 1 threads
Simulation is complete, simulated time: 1 us


Simulation Summary:
  Simulation Input File: /lus/bnchlu1/flash/stonea/experiment/scale_model_size/benchmarks/phold/phold_dist.py
  Ranks:                 1
  Simulated time:        1 us
  Threads:               1


Simulation Resource Utilization for Code Regions:
■ total
│ ├── Duration: 0.271762 s
│ └── Total Memory: 58.152 MB
├ ■ build
│ │ ├── Duration: 0.2292 s
│ │ └── Total Memory: 58.152 MB
│ ├ ■ graph-processing
│ │   ├── Duration: 0.167493 s
│ │   └── Total Memory: 39.72 MB
│ └ ■ construct
│     ├── Duration: 0.0615761 s
│     └── Total Memory: 58.152 MB
└ ■ execute
  │ ├── Duration: 0.04177 s
  │ └── Total Memory: 58.152 MB
  ├ ■ init
  │   ├── Duration: 0.000396013 s
  │   └── Total Memory: 58.152 MB
  ├ ■ setup
  │   ├── Duration: 0.000405073 s
  │   └──

> SST_CONFIG_FILE_PATH=/lus/bnchlu1/flash/stonea/experiment/scale_model_size/benchmarks/phold/sstsimulator.conf e4s-cl launch --image /lus/bnchlu1/flash/stonea/experiment/scale_model_size/hpc-ai-adv-dev-sst-core-master-latest.sif srun -N 1 --job-name=phold_961  -- sst --print-timing-info=3 --parallel-load=SINGLE ./phold_dist.py -- --width 31 --height 31
> SST_CONFIG_FILE_PATH=/lus/bnchlu1/flash/stonea/experiment/scale_model_size/benchmarks/phold/sstsimulator.conf e4s-cl launch --image /lus/bnchlu1/flash/stonea/experiment/scale_model_size/hpc-ai-adv-dev-sst-core-master-latest.sif srun -N 1 --job-name=phold_1936  -- sst --print-timing-info=3 --parallel-load=SINGLE ./phold_dist.py -- --width 44 --height 44
> SST_CONFIG_FILE_PATH=/lus/bnchlu1/flash/stonea/experiment/scale_model_size/benchmarks/phold/sstsimulator.conf e4s-cl launch --image /lus/bnchlu1/flash/stonea/experiment/scale_model_size/hpc-ai-adv-dev-sst-core-master-latest.sif srun -N 1 --job-name=phold_2916  -- sst --print-timing-in

## Watch squeue

In [ ]:
watch_queue_widget()

## Inspect results

In [14]:
inspect_logs(f'{baseDir}/runs')

/lus/bnchlu1/flash/stonea/experiment/scale_model_size/runs/size_961
/lus/bnchlu1/flash/stonea/experiment/scale_model_size/runs/size_1936
/lus/bnchlu1/flash/stonea/experiment/scale_model_size/runs/size_2916
/lus/bnchlu1/flash/stonea/experiment/scale_model_size/runs/size_3969
/lus/bnchlu1/flash/stonea/experiment/scale_model_size/runs/size_4900



# Preprocess

In [15]:
import os, glob

fullpath = f"{baseDir}/runs"
print(f'\n===== running extract under {fullpath} =====')
cd(fullpath)

data = extract_sst_output_in_files(sorted(glob.glob("size_*")))
csv_lines = convert_to_csv(data)
csv_name = f"{baseDir}/runs/results.csv"

with open(csv_name, 'w') as f:
    f.write('\n'.join(csv_lines))

if os.path.exists(csv_name):
    with open(csv_name, 'r') as f:
        print(f'\n===== {csv_name} =====')
        print(f.read())
else:
    print(f'\n===== {csv_name} (not created) =====')


===== running extract under /lus/bnchlu1/flash/stonea/experiment/scale_model_size/runs =====
  total_duration: not found
  total_memory: not found
  build_duration: not found
  build_memory: not found
  graph_processing_duration: not found
  graph_processing_memory: not found
  construct_duration: not found
  construct_memory: not found
  execute_time: not found
  execute_memory: not found
  init_duration: not found
  init_memory: not found
  setup_duration: not found
  setup_memory: not found
  run_duration: not found
  run_memory: not found
  complete_duration: not found
  complete_memory: not found
  finish_duration: not found
  finish_memory: not found
  global_active_activities: not found
  total_duration: not found
  total_memory: not found
  build_duration: not found
  build_memory: not found
  graph_processing_duration: not found
  graph_processing_memory: not found
  construct_duration: not found
  construct_memory: not found
  execute_time: not found
  execute_memory: not fo

# Plot

In [16]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

try:
    df = pd.read_csv(f"{baseDir}/runs/results.csv")
except FileNotFoundError as e:
    print(f'ERROR: File not found - {e.filename}')
    raise StopExecution()

fig = plt.figure()
ax = fig.add_subplot(111)

plot_value='total_duration'
ylabel = 'Total duration (secs)'

x = df["Size"].to_numpy(dtype=float)
y = df[plot_value].to_numpy(dtype=float)

# Fit y = slope*x + intercept using least squares
slope, intercept = np.polyfit(x, y, 1)
y_fit = slope * x + intercept

# Compute coefficient of determination (R^2)
ss_res = np.sum((y - y_fit) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r_squared = 1.0 - (ss_res / ss_tot if ss_tot != 0 else 0.0)

print(f"Slope: {slope:.6g}")
print(f"R-squared: {r_squared:.6f}")

ax.scatter(x=x, y=y, c='b', marker="s", label='Observed')
ax.plot(x, y_fit, c='crimson', linewidth=2, label='Linear fit')
ax.legend()

fit_text = f"y = {slope:.3g}x + {intercept:.3g}\n$R^2$ = {r_squared:.4f}"
ax.text(
    0.02, 0.98, fit_text,
    transform=ax.transAxes,
    va='top', ha='left',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85)
 )

plt.title(f'SST {benchmarkPath} single-node component scaling ({plot_value})')
plt.xlabel('Number of components')
plt.ylabel(ylabel)
plt.show()

ModuleNotFoundError: No module named 'pyparsing'